# Динамічний тренажер: переріз циліндра та розгортка

Модель поєднує побудову перерізу бічної поверхні циліндра січною площиною з його розгорткою.

Циліндр задається радіусом `R`, висотою `H` і центром основи `(cx, cy, cz)`. Площина:

\[
z=cz+a(x-cx)+b(y-cy)+d.
\]

На бічній поверхні циліндра:

\[
x=cx+R\cos\varphi,\quad y=cy+R\sin\varphi.
\]

Отже, лінія перерізу задається функцією:

\[
z(\varphi)=cz+d+aR\cos\varphi+bR\sin\varphi.
\]

Для розгортки кутова координата переходить у лінійну: `u = R·φ`, а вертикальна координата залишається `v = z`. Тому та сама крива на розгортці має координати `(u, z)`.

**Змінюємо положення або нахил площини — одночасно оновлюються 3D-переріз і його розгортка.**


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def section_and_development(cx=0., cy=0., cz=0., R=2., H=5.,
                            a=.35, b=.15, d=2.5, show_generators=True):
    phi = np.linspace(0, 2*np.pi, 1400)
    u = R*phi
    z = cz + d + a*R*np.cos(phi) + b*R*np.sin(phi)
    valid = (z >= cz-1e-9) & (z <= cz+H+1e-9)
    z_line = np.where(valid, z, np.nan)

    fig = plt.figure(figsize=(15, 6.5))
    ax3 = fig.add_subplot(131, projection="3d")
    axnet = fig.add_subplot(132)
    axproj = fig.add_subplot(133)

    # Циліндрична поверхня
    ph = np.linspace(0, 2*np.pi, 100)
    zz = np.linspace(cz, cz+H, 45)
    P, Z = np.meshgrid(ph, zz)
    X, Y = cx+R*np.cos(P), cy+R*np.sin(P)
    ax3.plot_surface(X, Y, Z, color="lightsteelblue", alpha=.28, linewidth=0)
    for z0 in (cz, cz+H):
        ax3.plot(cx+R*np.cos(ph), cy+R*np.sin(ph),
                 np.full_like(ph,z0), color="steelblue", lw=1.3)

    # Січна площина, обрізана межами циліндра за висотою
    gx = np.linspace(cx-R, cx+R, 45)
    gy = np.linspace(cy-R, cy+R, 45)
    GX, GY = np.meshgrid(gx, gy)
    GZ = cz+d+a*(GX-cx)+b*(GY-cy)
    mask = ((GX-cx)**2+(GY-cy)**2 <= R**2) & (GZ >= cz) & (GZ <= cz+H)
    ax3.plot_surface(GX, GY, np.where(mask,GZ,np.nan),
                     color="orange", alpha=.2, linewidth=0)

    xcut, ycut = cx+R*np.cos(phi), cy+R*np.sin(phi)
    ax3.plot(xcut, ycut, z_line, color="crimson", lw=3)
    if show_generators:
        for ang in np.linspace(0,2*np.pi,12,endpoint=False):
            xx, yy = cx+R*np.cos(ang), cy+R*np.sin(ang)
            ax3.plot([xx,xx],[yy,yy],[cz,cz+H],color="gray",lw=.55,alpha=.45,ls=":")
    ax3.set_title("3D: циліндр і переріз")
    ax3.set_xlabel("X"); ax3.set_ylabel("Y"); ax3.set_zlabel("Z")
    ax3.set_box_aspect((1,1,max(H/(2*R),.5)))
    ax3.view_init(elev=23,azim=-55)

    # Прямокутна розгортка бічної поверхні
    circumference = 2*np.pi*R
    axnet.plot([0,circumference,circumference,0,0],
               [cz,cz,cz+H,cz+H,cz],color="black",lw=1.5)
    axnet.plot(u,z_line,color="crimson",lw=2.8)
    axnet.set_xlim(0,circumference)
    axnet.set_ylim(cz-.08*H,cz+1.08*H)
    axnet.set_xlabel("u = R·φ — відстань уздовж кола")
    axnet.set_ylabel("v = z")
    axnet.set_title("Розгортка бічної поверхні")
    axnet.grid(True,alpha=.25)

    # Фронтальна проєкція циліндра та лінії перерізу
    axproj.plot([cx-R,cx-R],[cz,cz+H],color="steelblue",lw=1.4)
    axproj.plot([cx+R,cx+R],[cz,cz+H],color="steelblue",lw=1.4)
    axproj.plot([cx-R,cx+R],[cz,cz],color="steelblue",lw=1.4)
    axproj.plot([cx-R,cx+R],[cz+H,cz+H],color="steelblue",lw=1.4)
    axproj.scatter(xcut[valid],z[valid],s=5,color="crimson")
    axproj.set_xlabel("X"); axproj.set_ylabel("Z")
    axproj.set_title("Фронтальна проєкція")
    axproj.grid(True,alpha=.25)
    axproj.set_aspect("equal",adjustable="datalim")

    zmin,zmax=float(np.min(z)),float(np.max(z))
    if np.all(valid):
        print("Переріз повністю лежить на бічній поверхні.")
    elif not np.any(valid):
        print("Площина не перетинає бічну поверхню в межах висоти циліндра.")
    else:
        print("Переріз частково виходить на межі циліндра (верхню/нижню основу).")
    print(f"Висоти перетину: {zmin:.2f} … {zmax:.2f}; межі циліндра: {cz:.2f} … {cz+H:.2f}")
    print(f"Ширина розгортки: 2πR = {circumference:.2f}")

    plt.tight_layout()
    plt.show()

interact(
    section_and_development,
    cx=FloatSlider(min=-2,max=2,step=.25,value=0,description="cx"),
    cy=FloatSlider(min=-2,max=2,step=.25,value=0,description="cy"),
    cz=FloatSlider(min=-2,max=2,step=.25,value=0,description="cz"),
    R=FloatSlider(min=1,max=4,step=.25,value=2,description="R"),
    H=FloatSlider(min=2,max=8,step=.25,value=5,description="H"),
    a=FloatSlider(min=-3,max=1.5,step=.05,value=.50,description="Нахил a"),
    b=FloatSlider(min=-1.5,max=1.5,step=.05,value=.15,description="Нахил b"),
    d=FloatSlider(min=-2,max=9,step=.1,value=2.5,description="Зсув d"),
    show_generators=Checkbox(value=True,description="Твірні")
)


interactive(children=(FloatSlider(value=0.0, description='cx', max=2.0, min=-2.0, step=0.25), FloatSlider(valu…

<function __main__.section_and_development(cx=0.0, cy=0.0, cz=0.0, R=2.0, H=5.0, a=0.35, b=0.15, d=2.5, show_generators=True)>

## Практичні завдання

1. **Горизонтальний переріз:** `a = 0`, `b = 0`. Змінюйте `d`. Порівняйте коло на циліндрі з горизонтальною прямою на розгортці.
2. **Нахил в одному напрямку:** задайте `a ≠ 0`, `b = 0`. Простежте за кривою на розгортці.
3. **Зміна напряму нахилу:** змініть знак `a` або `b`.
4. **Частковий переріз:** переміщуйте площину параметром `d`, щоб вона перетинала верхню або нижню межу циліндра.
5. Поясніть, чому ширина розгортки дорівнює довжині кола `2πR`.

## Геометричний зміст

- На циліндрі крива визначається кутом `φ` та висотою `z`.
- Під час розгортання поверхні довжина дуги `Rφ` стає горизонтальною координатою `u`.
- Вертикальна координата не змінюється.
- Тому точки перерізу на 3D-поверхні та на розгортці відповідають одна одній.

**Обмеження цієї версії:** будується переріз саме бічної поверхні. Якщо площина перетинає диски основ, їхні контури перетину поки не додаються до розгортки. Вертикальний шов розгортки розташовано при `φ = 0`.
